In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from loaders._load_vn30_reg import preprocess, VN30, TARGETS
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import r2_score, mean_absolute_percentage_error

In [5]:
track = {"r2": [], "mape": []}
for symbol in VN30:
    data = preprocess(symbol, lag=2)
    X_train, Y_train = data['train']
    X_val, Y_val = data['val']
    X_test, Y_test = data['test']
    target_scaler = data['scaler']['target']

    tscv = TimeSeriesSplit(n_splits=3)
    model = RandomizedSearchCV(
        estimator=RandomForestRegressor(n_jobs=-1, random_state=42),
        param_distributions={
            "n_estimators": [10, 20, 50],
            "max_depth": [3, 5, 7, 9],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4]
        }, 
        cv=tscv, 
        n_iter=10, 
        random_state=42
    )

    model.fit(X_train, Y_train)

    Y_pred = model.predict(X_test)
    Y_pred = target_scaler.inverse_transform(Y_pred)
    Y_test = target_scaler.inverse_transform(Y_test)

    r2 = r2_score(Y_test, Y_pred)
    mape = mean_absolute_percentage_error(Y_test, Y_pred) * 100

    track["r2"].append(r2)
    track["mape"].append(mape)

    print(f"Symbol: {symbol}, R^2: {r2:.4f}, MAPE: {mape:.4f}")

print(f"Mean R^2: {np.mean(track['r2']):.4f}, Mean MAPE: {np.mean(track['mape']):.4f}")
print(f"Std R^2: {np.std(track['r2']):.4f}, Std MAPE: {np.std(track['mape']):.4f}")

Symbol: ACB, R^2: -10.1362, MAPE: 16.7889
Symbol: BCM, R^2: 0.9619, MAPE: 1.2542
Symbol: BID, R^2: -4.2238, MAPE: 10.4430
Symbol: BVH, R^2: 0.9839, MAPE: 0.9041
Symbol: CTG, R^2: 0.5565, MAPE: 3.0434
Symbol: FPT, R^2: -3.7318, MAPE: 29.8457
Symbol: GAS, R^2: 0.9594, MAPE: 0.7681
Symbol: GVR, R^2: 0.9733, MAPE: 1.5320
Symbol: HDB, R^2: -2.9628, MAPE: 17.0510
Symbol: HPG, R^2: 0.8316, MAPE: 1.4610
Symbol: LPB, R^2: -2.0205, MAPE: 34.9358
Symbol: MBB, R^2: 0.1734, MAPE: 4.2853
Symbol: MSN, R^2: 0.9494, MAPE: 1.3057
Symbol: MWG, R^2: 0.9820, MAPE: 1.2171
Symbol: PLX, R^2: 0.9799, MAPE: 1.1053
Symbol: SAB, R^2: 0.7006, MAPE: 2.0379
Symbol: SHB, R^2: 0.9648, MAPE: 0.9734
Symbol: SSB, R^2: 0.8072, MAPE: 2.1941
Symbol: SSI, R^2: 0.9338, MAPE: 1.2099
Symbol: STB, R^2: 0.8090, MAPE: 2.6105
Symbol: TCB, R^2: 0.9725, MAPE: 1.2556
Symbol: TPB, R^2: 0.9322, MAPE: 1.3658
Symbol: VCB, R^2: 0.5945, MAPE: 1.2947
Symbol: VHM, R^2: 0.9364, MAPE: 1.8934
Symbol: VIB, R^2: 0.9189, MAPE: 1.1519
Symbol: VIC, R